In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, root_mean_squared_error
from sklearn.model_selection import train_test_split

In [2]:
HERE = Path.cwd()
DATA = HERE / "data" / "uber.csv"
if not DATA.exists():
    DATA = HERE / "uber.csv"

In [3]:
df = pd.read_csv(DATA).dropna()
df = df[(df.fare_amount > 0) & df.passenger_count.between(1, 6)]
df = df[df.pickup_longitude.between(-75, -72) & df.dropoff_longitude.between(-75, -72)]
df = df[df.pickup_latitude.between(40, 42) & df.dropoff_latitude.between(40, 42)]

In [4]:
lat1, lon1 = np.radians(df.pickup_latitude), np.radians(df.pickup_longitude)
lat2, lon2 = np.radians(df.dropoff_latitude), np.radians(df.dropoff_longitude)
a = np.sin((lat2 - lat1) / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin((lon2 - lon1) / 2) ** 2
df["distance_km"] = 6371 * 2 * np.arcsin(np.sqrt(a))
df["hour_of_day"] = pd.to_datetime(df.pickup_datetime).dt.hour
df = df[["distance_km", "passenger_count", "hour_of_day", "fare_amount"]]

In [5]:
q1, q3 = df.fare_amount.quantile([0.25, 0.75])
outliers = ~df.fare_amount.between(q1 - 1.5 * (q3 - q1), q3 + 1.5 * (q3 - q1))
print("Outliers removed:", outliers.sum())
df = df[~outliers]

Outliers removed: 16708


In [6]:
print(df.corr().round(2))

                 distance_km  passenger_count  hour_of_day  fare_amount
distance_km             1.00            -0.00        -0.03         0.71
passenger_count        -0.00             1.00         0.02         0.01
hour_of_day            -0.03             0.02         1.00        -0.01
fare_amount             0.71             0.01        -0.01         1.00


In [7]:
X, y = df.drop(columns="fare_amount"), df.fare_amount
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [8]:
models = {"Linear Regression": LinearRegression(), "Random Forest": RandomForestRegressor(random_state=42)}
for name, model in models.items():
    pred = model.fit(X_train, y_train).predict(X_test)
    print(f"{name}: R2 = {r2_score(y_test, pred):.3f}, RMSE = {root_mean_squared_error(y_test, pred):.3f}")

Linear Regression: R2 = 0.424, RMSE = 3.142
Random Forest: R2 = 0.627, RMSE = 2.530
